In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv(
    "../data/MachineLearningRating_v3.txt",
    sep="|"
)

In [ ]:
df["VehicleAge"] = 2015 - df["RegistrationYear"]

In [ ]:
df["LossRatio"] = np.where(
    df["TotalPremium"] > 0,
    df["TotalClaims"] / df["TotalPremium"],
    0
)

In [ ]:
df["HasClaim"] = np.where(
    df["TotalClaims"] > 0,
    1,
    0
)

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

In [ ]:
X = df_encoded.drop(columns=["TotalClaims"])

y = df_encoded["TotalClaims"]

In [ ]:
X = df_encoded.drop(columns=[
    "TotalClaims",
    "LossRatio",
    "Margin",
    "HasClaim"
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lr = LinearRegression()

lr.fit(X_train, y_train)

In [ ]:
lr_preds = lr.predict(X_test)

In [ ]:
from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

In [ ]:
lr_rmse = np.sqrt(
    mean_squared_error(y_test, lr_preds)
)

print(lr_rmse)

In [ ]:
lr_r2 = r2_score(y_test, lr_preds)

print(lr_r2)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
rf_preds = rf.predict(X_test)

In [ ]:
rf_rmse = np.sqrt(
    mean_squared_error(y_test, rf_preds)
)

rf_r2 = r2_score(y_test, rf_preds)

print(rf_rmse)
print(rf_r2)

In [ ]:
from xgboost import XGBRegressor

In [ ]:
xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb.fit(X_train, y_train)

In [ ]:
xgb_preds = xgb.predict(X_test)

In [ ]:
xgb_rmse = np.sqrt(
    mean_squared_error(y_test, xgb_preds)
)

xgb_r2 = r2_score(y_test, xgb_preds)

print(xgb_rmse)
print(xgb_r2)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],
    "RMSE": [
        lr_rmse,
        rf_rmse,
        xgb_rmse
    ],
    "R2": [
        lr_r2,
        rf_r2,
        xgb_r2
    ]
})

comparison

In [ ]:
import shap

In [ ]:
explainer = shap.Explainer(xgb)

In [ ]:
shap_values = explainer(X_test)

In [ ]:
shap.plots.beeswarm(shap_values)